# 📖 Notebook 1: Data Classification

Before you can protect personal data, you need to **find** it. In this notebook, we'll scan a database to discover PII (Personally Identifiable Information), classify each column by sensitivity level, and build a classification registry.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to scan database schemas to detect PII columns
- The four classification levels: Public, Internal, Confidential, Restricted
- How to build automated PII detection using pattern matching
- How to maintain a data classification registry
- Why unclassified data is treated as Restricted by default

## 🛠️ Setup

Start the infrastructure first:

```bash
cd enterprise-patterns/privacy-review
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `privacy_review`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import json
import re
from datetime import datetime

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "privacy_review",
    "user": "demo",
    "password": "demo"
}

# Redis connection settings
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

def get_redis_client():
    return redis.Redis(**REDIS_CONFIG)

# Test both connections
try:
    conn = get_db_connection()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker-compose up -d")

try:
    r = get_redis_client()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker-compose up -d")

## 🔍 Step 1: Discover All Tables and Columns

The first step in any privacy review is understanding **what data you have**. Most engineers are surprised by how much PII is scattered across their database.

We'll query PostgreSQL's `information_schema` to get every table and column in our database. This is the same approach real privacy scanners use.

In [ ]:
def discover_schema():
    """Query the database schema to find all tables and columns."""
    conn = get_db_connection()
    cursor = conn.cursor()

    cursor.execute("""
        SELECT table_name, column_name, data_type, is_nullable
        FROM information_schema.columns
        WHERE table_schema = 'public'
        ORDER BY table_name, ordinal_position
    """)

    schema = {}
    for table, column, dtype, nullable in cursor.fetchall():
        if table not in schema:
            schema[table] = []
        schema[table].append({
            "column": column,
            "type": dtype,
            "nullable": nullable == "YES"
        })

    conn.close()
    return schema

schema = discover_schema()

print("📋 Database Schema Discovery")
print("=" * 60)
for table, columns in schema.items():
    print(f"\n📁 {table} ({len(columns)} columns)")
    for col in columns:
        print(f"   ├── {col['column']:<30} {col['type']}")

## 🏷️ Step 2: Automated PII Detection

Now we need to figure out which of these columns contain personal data. We'll build a **PII scanner** that uses two approaches:

1. **Column name pattern matching** — column names like `email`, `ssn`, `phone` are obvious PII
2. **Data content sampling** — check actual values for patterns like email addresses or phone numbers

Real tools like Microsoft Purview, AWS Macie, or Google DLP use machine learning for this, but pattern matching gets you 80% of the way there.

In [ ]:
# PII detection rules based on column names
# Each rule maps a pattern to a classification level and PII type

PII_COLUMN_PATTERNS = {
    # RESTRICTED — highest sensitivity
    "restricted": {
        "government_id": ["ssn", "social_security", "tax_id", "national_id", "passport"],
        "financial":     ["card_number", "account_number", "routing_number", "credit_card"],
        "health":        ["diagnosis", "medical_record", "health_condition", "prescription"],
        "credentials":   ["password", "password_hash", "secret", "token"],
    },
    # CONFIDENTIAL — identifies a person
    "confidential": {
        "name":     ["first_name", "last_name", "full_name", "display_name", "shipping_name"],
        "email":    ["email", "email_address", "contact_email"],
        "phone":    ["phone", "phone_number", "mobile", "telephone"],
        "address":  ["street_address", "address", "shipping_address", "billing_address",
                     "zip_code", "postal_code"],
        "dob":      ["date_of_birth", "birth_date", "dob", "birthday"],
        "location": ["geo_city", "geo_lat", "geo_lon", "latitude", "longitude"],
        "network":  ["ip_address", "ip", "mac_address"],
    },
    # INTERNAL — company use only
    "internal": {
        "location":  ["city", "state", "country", "region", "geo_country"],
        "device":    ["user_agent", "device_type", "browser"],
        "metadata":  ["account_status", "signup_source", "created_at", "updated_at"],
    }
}

def classify_column_by_name(column_name):
    """Classify a column based on its name matching known PII patterns."""
    col_lower = column_name.lower()

    for level in ["restricted", "confidential", "internal"]:
        for pii_type, patterns in PII_COLUMN_PATTERNS[level].items():
            for pattern in patterns:
                if pattern in col_lower or col_lower in pattern:
                    return level, pii_type

    return "public", None

# Test our classifier on a few examples
test_columns = ["email", "ssn", "first_name", "ip_address", "product_id", "price", "user_agent"]

print("🧪 Column Name Classification Test")
print("=" * 55)
for col in test_columns:
    level, pii_type = classify_column_by_name(col)
    icons = {"restricted": "🔴", "confidential": "🟡", "internal": "🔵", "public": "🟢"}
    pii_label = f" ({pii_type})" if pii_type else ""
    print(f"  {icons[level]} {col:<25} → {level}{pii_label}")

## 📊 Step 3: Content-Based PII Scanning

Column names don't always tell the full story. A column called `description` might contain PII that users typed in (like "My email is john@gmail.com"). We need to **scan the actual data** too.

This is especially important for **free-text fields** like support tickets, comments, and notes.

In [ ]:
# Content-based PII detection using regular expressions
# These patterns match common PII formats in text

PII_CONTENT_PATTERNS = {
    "email": {
        "regex": r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
        "classification": "confidential",
        "description": "Email address found in text"
    },
    "ssn": {
        "regex": r'\b\d{3}-\d{2}-\d{4}\b',
        "classification": "restricted",
        "description": "SSN pattern (XXX-XX-XXXX) found in text"
    },
    "phone_us": {
        "regex": r'\b(?:\+?1[-.]?)?\(?\d{3}\)?[-.]?\d{3}[-.]?\d{4}\b',
        "classification": "confidential",
        "description": "US phone number found in text"
    },
    "credit_card": {
        "regex": r'\b(?:4\d{3}|5[1-5]\d{2}|6011|3[47]\d{2})[-\s]?\d{4}[-\s]?\d{4}[-\s]?\d{4}\b',
        "classification": "restricted",
        "description": "Credit card number found in text"
    },
    "dob": {
        "regex": r'\b(?:DOB|date of birth|born on|birthday)[:\s]+\d{1,2}[/\-]\d{1,2}[/\-]\d{2,4}\b',
        "classification": "restricted",
        "description": "Date of birth mentioned in text"
    },
    "ssn_last4": {
        "regex": r'(?:SSN|social security)\s*(?:last\s*4|ending)[:\s]*\d{4}',
        "classification": "restricted",
        "description": "SSN reference found in text"
    }
}

def scan_text_for_pii(text):
    """Scan a text string for PII patterns. Returns list of findings."""
    if not text:
        return []

    findings = []
    for pii_type, config in PII_CONTENT_PATTERNS.items():
        matches = re.findall(config["regex"], text, re.IGNORECASE)
        if matches:
            findings.append({
                "type": pii_type,
                "classification": config["classification"],
                "matches": matches,
                "description": config["description"]
            })
    return findings

# Test on our support ticket data — this is where PII often hides
print("🔎 Scanning Support Tickets for Hidden PII")
print("=" * 60)

conn = get_db_connection()
cursor = conn.cursor()
cursor.execute("SELECT id, subject, description, internal_notes FROM support_tickets")

for ticket_id, subject, description, notes in cursor.fetchall():
    print(f"\n📝 Ticket #{ticket_id}: {subject}")

    # Scan the description field
    desc_findings = scan_text_for_pii(description)
    if desc_findings:
        print(f"   ⚠️  PII in description:")
        for f in desc_findings:
            icon = "🔴" if f["classification"] == "restricted" else "🟡"
            print(f"      {icon} {f['description']} — found: {f['matches']}")

    # Scan internal notes
    notes_findings = scan_text_for_pii(notes)
    if notes_findings:
        print(f"   ⚠️  PII in internal_notes:")
        for f in notes_findings:
            icon = "🔴" if f["classification"] == "restricted" else "🟡"
            print(f"      {icon} {f['description']} — found: {f['matches']}")

    if not desc_findings and not notes_findings:
        print("   ✅ No PII patterns detected")

conn.close()

## 🗂️ Step 4: Full Database Classification Scan

Let's combine both approaches — column name matching AND content scanning — to classify every column in every table. This gives us a complete picture of where PII lives in our database.

This is what a real privacy scanner does when you first connect it to a database.

In [ ]:
def full_database_scan():
    """Scan the entire database and classify every column."""
    schema = discover_schema()
    results = []

    # Skip metadata/registry tables — we only want application tables
    skip_tables = {
        "data_classification_registry", "privacy_impact_assessments",
        "data_retention_policies", "purge_audit_log"
    }

    conn = get_db_connection()
    cursor = conn.cursor()

    for table, columns in schema.items():
        if table in skip_tables:
            continue

        for col_info in columns:
            col_name = col_info["column"]
            col_type = col_info["type"]

            # Step 1: Classify by column name
            level, pii_type = classify_column_by_name(col_name)

            # Step 2: For text columns, also sample content
            content_pii = []
            if col_type in ("text", "character varying") and level == "public":
                try:
                    cursor.execute(
                        f'SELECT "{col_name}" FROM "{table}" WHERE "{col_name}" IS NOT NULL LIMIT 50'
                    )
                    for (value,) in cursor.fetchall():
                        findings = scan_text_for_pii(str(value))
                        content_pii.extend(findings)
                except Exception:
                    pass

            # Upgrade classification if content scan found PII
            if content_pii:
                worst = max(content_pii, key=lambda f: 
                    ["public", "internal", "confidential", "restricted"].index(f["classification"]))
                if ["public", "internal", "confidential", "restricted"].index(worst["classification"]) > \
                   ["public", "internal", "confidential", "restricted"].index(level):
                    level = worst["classification"]
                    pii_type = "free_text"

            results.append({
                "table": table,
                "column": col_name,
                "type": col_type,
                "classification": level,
                "pii_type": pii_type,
                "content_findings": len(content_pii)
            })

    conn.close()
    return results

# Run the full scan
scan_results = full_database_scan()

# Display results grouped by table
print("📊 Full Database Classification Report")
print("=" * 70)

icons = {"restricted": "🔴", "confidential": "🟡", "internal": "🔵", "public": "🟢"}
current_table = None

for r in scan_results:
    if r["table"] != current_table:
        current_table = r["table"]
        print(f"\n📁 {current_table}")

    pii_label = f" ({r['pii_type']})" if r["pii_type"] else ""
    content_flag = f" ⚠️ +{r['content_findings']} content matches" if r["content_findings"] else ""
    print(f"   {icons[r['classification']]} {r['column']:<30} {r['classification']}{pii_label}{content_flag}")

# Summary statistics
print("\n" + "=" * 70)
print("📈 Summary")
for level in ["restricted", "confidential", "internal", "public"]:
    count = sum(1 for r in scan_results if r["classification"] == level)
    print(f"   {icons[level]} {level.upper():<15} {count} columns")

## 💾 Step 5: Save Classifications to the Registry

Once we've classified everything, we need to **persist** the results so other teams can look up the classification of any column. We'll save to both:

1. **PostgreSQL** — the `data_classification_registry` table (source of truth)
2. **Redis** — a cache for fast lookups during API requests

In a real system, this registry is queried every time someone writes a query or builds an API endpoint — "am I allowed to return this column to this caller?"

In [ ]:
def save_classifications_to_db(scan_results):
    """Save scan results to the classification registry table."""
    conn = get_db_connection()
    cursor = conn.cursor()

    saved_count = 0
    for r in scan_results:
        # Only save columns that have PII (not public)
        if r["classification"] == "public":
            continue

        try:
            cursor.execute("""
                INSERT INTO data_classification_registry
                    (table_name, column_name, classification, pii_type, classified_by)
                VALUES (%s, %s, %s, %s, %s)
                ON CONFLICT (table_name, column_name) DO UPDATE SET
                    classification = EXCLUDED.classification,
                    pii_type = EXCLUDED.pii_type,
                    classified_at = CURRENT_TIMESTAMP
            """, (r["table"], r["column"], r["classification"], r["pii_type"], "auto-scanner"))
            saved_count += 1
        except Exception as e:
            print(f"   ⚠️ Error saving {r['table']}.{r['column']}: {e}")

    conn.commit()
    conn.close()
    return saved_count

def cache_classifications_in_redis(scan_results):
    """Cache classifications in Redis for fast lookups."""
    r = get_redis_client()
    pipe = r.pipeline()

    for result in scan_results:
        if result["classification"] == "public":
            continue

        key = f"classification:{result['table']}:{result['column']}"
        pipe.hset(key, mapping={
            "classification": result["classification"],
            "pii_type": result["pii_type"] or "",
            "scanned_at": datetime.now().isoformat()
        })
        # Classifications don't change often — cache for 24 hours
        pipe.expire(key, 86400)

    pipe.execute()

# Save to both PostgreSQL and Redis
saved = save_classifications_to_db(scan_results)
cache_classifications_in_redis(scan_results)

print(f"✅ Saved {saved} classifications to PostgreSQL registry")
print(f"✅ Cached classifications in Redis (24h TTL)")

# Demonstrate a Redis lookup
r = get_redis_client()
lookup = r.hgetall("classification:users:ssn")
print(f"\n🔍 Redis lookup for users.ssn: {lookup}")

## 🛡️ Step 6: Classification-Aware Query Helper

Now that we have a registry, let's build a helper that **checks classifications before returning data**. This is how access control works in practice — your API layer consults the registry and masks or blocks restricted columns.

For example, a support agent should see the customer's name but NOT their SSN.

In [ ]:
# Access levels for different roles
ROLE_ACCESS = {
    "public_api":     ["public"],
    "support_agent":  ["public", "internal", "confidential"],
    "data_engineer":  ["public", "internal"],
    "privacy_officer": ["public", "internal", "confidential", "restricted"],
}

def get_column_classification(table, column):
    """Look up a column's classification, checking Redis first."""
    r = get_redis_client()
    key = f"classification:{table}:{column}"
    cached = r.hgetall(key)

    if cached:
        return cached.get("classification", "public")

    # Fallback: unclassified data is treated as RESTRICTED (safe default)
    return "restricted"

def query_with_access_control(table, columns, role):
    """Query a table but mask columns the role isn't allowed to see."""
    allowed_levels = ROLE_ACCESS.get(role, ["public"])

    # Build column list, replacing restricted columns with masked values
    select_parts = []
    masked_columns = []

    for col in columns:
        classification = get_column_classification(table, col)
        if classification in allowed_levels:
            select_parts.append(f'"{col}"')
        else:
            select_parts.append(f"'[MASKED]' AS \"{col}\"")
            masked_columns.append((col, classification))

    query = f"SELECT {', '.join(select_parts)} FROM \"{table}\" LIMIT 5"

    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(query)
    rows = cursor.fetchall()
    conn.close()

    return rows, masked_columns

# Demo: Same query, different roles see different data
columns = ["first_name", "last_name", "email", "ssn", "city", "account_status"]

for role in ["public_api", "support_agent", "privacy_officer"]:
    print(f"\n👤 Role: {role}")
    print(f"   Access: {ROLE_ACCESS[role]}")
    print("-" * 80)

    rows, masked = query_with_access_control("users", columns, role)

    # Print header
    header = f"  {'first_name':<12} {'last_name':<12} {'email':<25} {'ssn':<13} {'city':<12} {'status'}"
    print(header)

    for row in rows:
        vals = [str(v)[:12] if v else "" for v in row[:2]]
        vals.append(str(row[2])[:24] if row[2] else "")
        vals.append(str(row[3])[:12] if row[3] else "")
        vals.append(str(row[4])[:12] if row[4] else "")
        vals.append(str(row[5]) if row[5] else "")
        print(f"  {vals[0]:<12} {vals[1]:<12} {vals[2]:<25} {vals[3]:<13} {vals[4]:<12} {vals[5]}")

    if masked:
        print(f"  🔒 Masked: {', '.join(f'{c} ({l})' for c, l in masked)}")

## 🎯 Key Takeaways

1. **PII is everywhere** — it's in obvious places (email column) and non-obvious places (free-text support tickets)
2. **Classification must be automated** — manual review doesn't scale to thousands of tables
3. **Two scanning approaches** — column name patterns catch structured PII, content scanning catches hidden PII
4. **Registry is the source of truth** — every team should be able to look up any column's classification
5. **Default to Restricted** — if you don't know a column's classification, assume the worst
6. **Access control uses classification** — different roles see different columns based on their classification

### What Microsoft Does

- **Microsoft Purview** scans databases, files, and APIs automatically to classify data
- Every Azure service must register its data assets in the classification catalog
- Unclassified data in production triggers an alert to the privacy team
- Classification labels flow through to access policies, encryption rules, and retention schedules

### Next Notebook

In **Notebook 2: Privacy Impact Assessment**, we'll use these classifications to evaluate the risk of a new feature before it ships.